<a href="https://colab.research.google.com/github/slakshika2026/CV-Portal-Backend/blob/main/214113T_Quiz4_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# 2. Load Dataset

df = pd.read_csv("PJM_Load_hourly.csv")

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime')

data = df['PJM_Load_MW'].values.reshape(-1, 1)


In [ ]:
# 3. Normalize Data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)


In [ ]:
# 4. Convert to Supervised Learning (Sliding Window)
lookback = 168
horizon = 24

lookback = 24 * 7   # 168 hours
horizon = 24        # next day

X = []
y = []

for i in range(len(data_scaled) - lookback - horizon):
    X.append(data_scaled[i:i+lookback])
    y.append(data_scaled[i+lookback:i+lookback+horizon].flatten())

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)



X shape: (32704, 168, 1)
y shape: (32704, 24)


In [ ]:
# 5. Train / Validation / Test Split
n = len(X)

train_end = int(0.7 * n)
val_end = int(0.85 * n)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]


In [ ]:
# 6. Build RNN Model (1 layer)
model = Sequential([
    SimpleRNN(32, input_shape=(lookback, 1)),
    Dense(horizon)
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# 7. Compile Model

model.compile(
    optimizer='adam',
    loss='mse'
)


In [ ]:
# 8. Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


In [ ]:
# 9. Train Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)


Epoch 1/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0030 - val_loss: 0.0026
Epoch 2/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 18s 25ms/step - loss: 0.0029 - val_loss: 0.0025
Epoch 3/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0028 - val_loss: 0.0027
Epoch 4/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0030 - val_loss: 0.0028
Epoch 5/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - loss: 0.0029 - val_loss: 0.0026
Epoch 6/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 17s 24ms/step - loss: 0.0029 - val_loss: 0.0024
Epoch 7/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - loss: 0.0028 - val_loss: 0.0027
Epoch 8/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - loss: 0.0030 - val_loss: 0.0025
Epoch 9/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - loss: 0.0028 - val_loss: 0.0024
Epoch 10/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - loss: 0.0028 - val_loss: 0.0025
Epoch 11/50
716/716 ━━━━━━━━━━━━━━━━━━━━ 19s 27ms/step - loss: 0.0028 - val_loss: 0.0023
Epoch 12/50
716/716 ━━━━━━━━━━